# Prueba final (20 %)

*   **Caso:** La red aérea centroamericana
*   **Curso:** BCD5105 Modelado matemático
*   **Profesor:** Jordy Alfaro Brenes
*   **Modalidad:** Grupal (los grupos de trabajo del curso), sincrónica vía Zoom
*   **Duración:** 90 minutos
*   **Materiales permitidos:** Google Colab con el cuaderno `PruebaFinal_celdas.ipynb` suministrado, calculadora, y los archivos `aeropuertos.csv` y `rutas.csv`.
*   **No permitido:** Inteligencia artificial generativa, incluidas las funciones de asistencia con IA de Colab, que deben permanecer desactivadas.
*   **Valor:** 100 puntos en total. 
    *   **Parte A** (ejecución y análisis en Colab): 60 pts.
    *   **Parte B** (interpretación de resultados y conceptos): 40 pts.
*   **Alumnos:** 
    - Siloé Campos
    - Jason Corrau
    - Gabriel Corrales
    - David Mora

---

## El caso: ustedes son el equipo de planificación de una aerolínea regional

Una aerolínea regional opera en Centroamérica. Su equipo de ciencia de datos (ustedes) recibió dos archivos con datos reales de la base abierta OpenFlights: `aeropuertos.csv`, con los 10 aeropuertos internacionales principales de la región y sus coordenadas, y `rutas.csv`, con las 29 rutas comerciales registradas entre ellos y sus distancias de gran círculo en kilómetros.

Toda la prueba usa estos datos. Cuando un ejercicio necesite información adicional (tarifas, costos, probabilidades), esta se declara explícitamente como supuesto del enunciado, porque no forma parte de la base de datos.

# 1 Parte A: ejecutar y analizar en Colab (60 puntos)

## A1. La ruta de Liberia a Roatán (15 pts)

La aerolínea quiere llevar carga de Liberia (LIR) a Roatán (RTB) usando solo vuelos que existen en `rutas.csv`. La Celda A1 construye la red completa de 10 aeropuertos y 29 rutas y ejecuta el algoritmo de Dijkstra. Su salida:

```text
Distancias minimas desde LIR (km):
LIR: 0
SJO: 161
MGA: 482
SAL: 496
PTY: 696
GUA: 699
TGU: 706
SAP: 751
RTB: 916
BZE: 959

Ruta mas corta LIR -> RTB: LIR - SAL - RTB (916 km , 1 escala(s))

**Se pide:**

*   **(a) (5 pts)** Un analista propone la ruta LIR–SAL–SAP–RTB. Usando las distancias de `rutas.csv` (SAL–SAP 255, SAP–RTB 178), calculen cuánto mide esa propuesta y expliquen, con las etiquetas de la salida, por qué el algoritmo no la eligió.
    - R/ La propuesta del analista de seguir la ruta LIR–SAL–SAP–RTB suma una distancia total de 929 km, resultado de combinar los tramos de LIR a SAL (496 km), de SAL a SAP (255 km) y de SAP a RTB (178 km), el algoritmo de Dijkstra no eligió esta opción porque, tal como lo indica la salida, la ruta más corta encontrada es LIR–SAL–RTB con una distancia de 916 km y una sola escala. Al ser 916 km menor que los 929 km de la propuesta, el algoritmo priorizó la opción óptima que minimiza la distancia total.

*   **(b) (4 pts)** La lista de distancias sale ordenada de menor a mayor. Expliquen por qué ese orden no es casualidad sino una propiedad de cómo trabaja Dijkstra (¿en qué orden va “cerrando” los nodos?).
    - R/ El orden de menor a mayor no es casualidad, sino el reflejo exacto de la lógica central del algoritmo de Dijkstra. Este método funciona explorando la red de manera progresiva, como una onda que se expande desde el punto de origen. En cada iteración, el algoritmo revisa todos los nodos "abiertos" (no visitados) y siempre selecciona para "cerrar" aquel que tiene la distancia acumulada más pequeña, al cerrarlo, decreta que ha encontrado la ruta definitiva más corta hacia ese destino, ya que, al no existir distancias negativas, sería imposible encontrar un atajo pasando por un nodo que actualmente está más lejos. Por lo tanto, Dijkstra va descubriendo, confirmando y cerrando los nodos en estricto orden ascendente según su lejanía; la lista de distancias que observas es literalmente la secuencia cronológica en la que el algoritmo fue procesando y finalizando cada aeropuerto.

*   **(c) (6 pts)** Tarea de ejecución: cambien el origen y destino a `"SJO"` y `"RTB"`, corran de nuevo y reporten la ruta, su distancia y sus escalas. Según `rutas.csv` no existe vuelo directo SJO–RTB: expliquen qué hace el aeropuerto de escala elegido “mejor” que las otras opciones de conexión.
    - R/ Al ejecutar el algoritmo con SJO como origen y RTB como destino, la salida indica que la ruta más corta es SJO - TGU - RTB, con una distancia total de 820 km y 1 escala (en Tegucigalpa), dado que no existe un vuelo directo entre San José (SJO) y Roatán (RTB) en rutas.csv, el algoritmo debe buscar un nodo intermedio (hub). La escala en TGU es "mejor" que las demás opciones disponibles porque la suma de las distancias de sus dos segmentos (SJO–TGU de 558 km + TGU–RTB de 262 km) resulta en el menor kilometraje total acumulado (820 km).
    
    Si evaluamos los otros aeropuertos que conectan tanto con SJO como con RTB, vemos que todos generan rutas más largas:
    
        -   Vía SAP: SJO–SAP (728 km) + SAP–RTB (178 km) = 906 km.

        -   Vía SAL: SJO–SAL (652 km) + SAL–RTB (420 km) = 1072 km.

        -   Vía GUA: SJO–GUA (855 km) + GUA–RTB (470 km) = 1325 km.
    
    Dijkstra evalúa todas estas combinaciones y elige TGU porque es el nodo de conexión que minimiza de forma absoluta la distancia de viaje entre el origen y el destino final.

## A2. Repartir la flota entre Panamá y Guatemala (15 pts)

Desde San José, las dos rutas más rentables son SJO–PTY (539 km según `rutas.csv`) y SJO–GUA (855 km). Supuestos del enunciado: cada vuelo redondo a Panamá deja $500 y consume 3 horas de flota; a Guatemala, $600 y 4 horas. Hay 60 horas semanales de flota y 18 salidas asignadas por el aeropuerto. La Celda A2 resuelve el problema con `scipy.optimize.linprog`. Su salida:

```text
Plan optimo: 12 vuelos a Panama y 6 a Guatemala por semana
Ganancia semanal maxima: 9,600 USD
Horas de flota usadas: 60 de 60
Salidas usadas: 18 de 18
Valor de una hora extra de flota (precio sombra): 100 USD
Valor de un slot extra de salida (precio sombra): 200 USD



**Se pide:**

*   **(a) (4 pts)** La salida dice “60 de 60” y “18 de 18”. ¿Qué significa que ambas restricciones
estén activas y qué tiene que ver con que ambos precios sombra sean positivos?
    - R/ Que la salida indique “60 de 60” y “18 de 18” significa que la aerolínea consumió la totalidad de sus recursos; es decir, agotó tanto el límite de horas de flota como el límite de salidas (slots) que tenía disponibles. En programación lineal, esto implica que ambas restricciones son "activas" o "vinculantes", ya que no hay ninguna holgura o sobrante, esto está directamente relacionado con el hecho de que ambos precios sombra sean positivos. El precio sombra representa el aumento en la ganancia total que se obtendría si se consiguiera una unidad adicional de ese recurso. Como ambos recursos están agotados y son el "cuello de botella" que frena el crecimiento de las ganancias, conseguir una hora extra o una salida extra permitiría operar más vuelos y aumentar las utilidades (en 100 USD y 200 USD, respectivamente). Si algún recurso hubiera sobrado, la restricción sería inactiva y su precio sombra sería de cero, ya que tener más de un recurso que ya está sobrando no aporta ningún beneficio económico adicional.

*   **(b) (5 pts)** La gerencia puede comprar una hora más de flota o negociar un slot más de salida,
al mismo costo. Usando los precios sombra, recomienden cuál negociar y expliquen qué
mide exactamente un precio sombra.
    - R/ Se recomienda a la gerencia negociar un slot de salida adicional, el precio sombra de una salida extra es de 200 USD, mientras que el de una hora extra de flota es de 100 USD. Dado que ambas opciones tienen el mismo costo de adquisición, conseguir un slot adicional aportará el doble de ganancia a la aerolínea en comparación con obtener una hora extra. Un precio sombra mide el valor marginal de un recurso que se encuentra agotado, específicamente, indica exactamente en cuántos dólares aumentaría la ganancia total (la función objetivo) si la disponibilidad de ese recurso particular se incrementara en exactamente una unidad, asumiendo que el resto de las condiciones del modelo se mantienen constantes.

*   **(c) (6 pts)** Tarea de ejecución: cambien HORAS = 60 por HORAS = 70 y corran. Reporten
el nuevo plan y la nueva ganancia, y expliquen por qué el plan se volcó hacia Guatemala: ¿qué recurso dejó de ser el cuello de botella dominante y por qué eso favorece a
la ruta que más horas consume?
    - R/ Al correr el modelo con HORAS = 70, el nuevo plan óptimo es realizar 2 vuelos a Panamá y 16 vuelos a Guatemala por semana, alcanzando una ganancia semanal máxima de 10,600 USD, el plan se volcó drásticamente hacia Guatemala porque, al inyectar 10 horas adicionales, el tiempo de flota se relajó y dejó de ser la restricción más asfixiante en proporción a la cantidad de salidas. Anteriormente con 60 horas, el modelo priorizaba Panamá porque consumía menos tiempo (3 horas), lo que permitía exprimir al máximo las 60 horas sin agotar los 18 slots. Al aumentar el tiempo disponible a 70 horas, la aerolínea cuenta con casi 3.9 horas de flota disponibles por cada slot asignado (70/18). Dado que el límite de salidas se vuelve la restricción operativa principal (el nuevo cuello de botella dominante), al modelo le conviene asignar cada valioso slot al vuelo que genera más dinero por salida: Guatemala (600 USD frente a los 500 USD de Panamá). Ahora hay tiempo suficiente para "pagar" las 4 horas que requiere la ruta guatemalteca sin sacrificar slots.
    

## A3. ¿Abrir la ruta San José–Belice? (15 pts)
En rutas.csv no existe vuelo directo SJO–BZE. La gerencia evalúa abrirlo. Supuestos del
enunciado sobre la utilidad del primer año (miles de $): demanda alta +180 con probabilidad
0.40; media +40 con 0.35; baja −120 con 0.25. No abrir vale 0 con certeza. La Celda A3 simula
10 000 “primeros años” por Monte Carlo. Su salida:

```text
Simulacion de 10000 'primeros anios' de la ruta SJO -BZE:
utilidad promedio si ABRE la ruta: +55.0 mil USD
utilidad si NO abre: +0.0 mil USD (certeza)
porcentaje de anios con perdida: 25.2%
peor resultado posible: -120 mil USD
utilidad promedio decidiendo CON el estudio previo: +85.2 mil USD
valor de la informacion ( diferencia de promedios ): 30.3 mil USD



**Se pide:**

*   **(a) (4 pts)** El promedio simulado (+55.0) recomienda abrir, pero un 25% de los años simulados pierde dinero. Expliquen qué decisor seguiría el promedio y qué decisor se fijaría en
el peor caso (−120), y qué criterio del curso representa cada uno.
    -   R/ El promedio simulado de +55.0 mil USD sugiere que en términos de valor esperado, abrir la ruta es acertada. Un decisor que sigue el criterio del valor esperado elegiría abrir la ruta, ya que el beneficio promedio supera a la opción de no hacer nada (0), por otro lado, un decisor que aplica el criterio maximin (o criterio del peor caso) se enfocaría en el resultado más adverso, que es una pérdida de 120 mil USD. Este decisor, siendo extremadamente conservador y averso al riesgo, rechazaría el proyecto para evitar la posibilidad de una pérdida, sin importar la probabilidad de que ocurra.

*   **(b) (5 pts)** La última línea dice que el estudio de mercado vale 30.3 mil $. Expliquen de
dónde sale ese valor: ¿qué hace distinto el decisor que tiene el estudio en el escenario
de demanda baja?
    -   R/ El valor de la información surge de la diferencia entre dos estrategias. Sin el estudio, el decisor debe decidir antes de conocer la demanda y aplica la misma acción (abrir) para todos los escenarios, lo que genera una pérdida en el 25% de los casos. Con el estudio, el decisor obtiene información sobre el escenario de demanda antes de decidir. Esto permite actuar de forma óptima en cada caso: si el estudio predice una demanda alta o media, decide abrir; pero si predice una demanda baja, decide no abrir, evitanda la pérdida de -120 mil USD y obteniendo 0 en ese escenario. El valor de 30.3 mil USD es, por lo tanto, la ganancia promedio adicional que se obtiene al tener la flexibilidad de “no abrir” en el escenario desfavorable.

*   **(c) (6 pts)** Tarea de ejecución: el área financiera propone el escenario pesimista [0.20,
0.30, 0.50]. Cámbienlo, corran y reporten: la nueva utilidad promedio, si la recomendación cambia, y qué pasó con el valor de la información. Expliquen por qué la información
vale más justamente cuando el panorama es más incierto o adverso.
    -   R/ La recomendación se invierte por completo: el promedio pasa de +55.0 a -13.9 mil USD, así que el criterio del valor esperado ahora recomienda no abrir la ruta. El riesgo de pérdida sube de 25.2% a cerca del 50% de los años simulados, coherente con que la probabilidad del escenario “baja” subió justamente a 0.50. El valor de la información también crece de 30.3 a 60.8 mil USD, casi el doble porque el panorama es más adverso: al ser mucho más probable la demanda baja, la opción de “no abrir” se vuelve crucial, y el estudio de mercado permite identificar ese escenario con mayor frecuencia y evitar la pérdida de 120 mil USD. Por eso la información vale más justamente cuando el panorama es más incierto o adverso: es ahí donde decidir a ciegas sale más caro.

## A4. ¿Cuál promoción mostrar? (15 pts)

El registro histórico del sitio web (300 visitas) muestra: promoción A con 58 compras en 200
vistas (tasa 0.290), B con 18 en 80 (0.225), C con 5 en 20 (0.250). Un analista propone quedarse
con A para siempre. Antes de decidir, el equipo simula la campaña completa con la Celda
A4, que compara al terco (muestra al que va ganando, con 10 % de pruebas al azar, siempre)
contra el apostador (apuesta según su corazonada, y explora menos conforme aprende). Las tasas
verdaderas, que nadie conoce, son A 0.25, B 0.18, C 0.40. Salida con 5 000 visitantes (promedio de
50 repeticiones):

```text
EL TERCO: 1939 compras en 5000 visitantes
% de visitantes que vio la MEJOR promocion (C): 91.9%
reparto: A 4%, B 4%, C 92%
EL APOSTADOR : 1978 compras en 5000 visitantes
% de visitantes que vio la MEJOR promocion (C): 97.6%
reparto: A 1%, B 1%, C 98%

## Se pide
*   **(a) (4 pts)** El registro histórico decía que A era la mejor, pero la verdadera mejor es C.
Expliquen por qué el registro engaña: ¿qué tiene de especial la evidencia sobre C (5
compras en 20 vistas) que la vuelve poco confiable, y qué habría pasado si se adopta la
propuesta del analista?
    -   R/ El problema de fondo es el tamaño de muestra detrás de cada tasa. La tasa de C
(0.250) se calculó con apenas 20 vistas y 5 compras, una muestra muy pequeña. Su
error estándar es = 0.097, lo que da un intervalo de confianza aproximado de [0.06,
0.44]: un rango muy grande que ni siquiera excluye la tasa verdadera (0.40). En cambio,
la tasa de A (0.290) se calculó con 200 vistas, diez veces más datos, y su error estándar
es mucho menor (=0.032), con un intervalo de [0.23, 0.35] bastante más angosto. El
registro "engaña" porque pone en igualdad dos números que no tienen la misma
confiabilidad estadística: 0.290 es una estimación sólida, mientras que 0.250 es
prácticamente ruido, pudo haber salido así por pura casualidad aunque la tasa real de C
fuera mucho más alta, como de hecho lo es (0.40).
La consecuencia de esto es que si se adopta la propuesta del analista de "quedarse con
A para siempre", la campaña deja de mostrar C por completo. Eso significa que nunca
se recolecta más evidencia sobre C, y por lo tanto nunca se corrige el error inicial, no
porque el error sea imposible de detectar, sino porque la política elegida (explotación
pura, sin exploración) elimina cualquier oportunidad de aprender. El resultado sería
quedarse atrapados indefinidamente convirtiendo al 0.25 en lugar del 0.40 real,
perdiendo de forma permanente cerca de 0.15 compras por cada visita que pudo
haberse convertido, y esa pérdida se acumula sin límite mientras más dure la campaña.

*   **(b) (5 pts)** Ambas políticas descubren que C es la mejor, pero el terco se queda en 91.9 % y
el apostador llega a 97.6 %. Expliquen la causa estructural de esa diferencia: ¿qué hace el
terco para siempre que el apostador deja de hacer?
    -   R/Ambas políticas sí logran descubrir que C es la mejor opción 91.9% de las visitas para
el terco, 97.6% para el apostador, así que la diferencia no está en si aprenden, sino en
qué tanto siguen "pagando" por explorar una vez que ya aprendieron. El terco tiene una
regla fija: 10% de las visitas siempre se reparten al azar entre las tres promociones, sin
importar cuánta evidencia acumulada tenga a favor de C. El terco arrastra un "impuesto
de exploración" constante y permanente, incluso mucho después de que ya está claro
cuál promoción es la mejor.
El apostador, en cambio, "explora menos conforme aprende": su probabilidad de
desviarse hacia A o B no es fija, sino que se va reduciendo a medida que acumula
confianza sobre cuál es la mejor opción. Al principio explora de forma parecida al terco,
pero con el tiempo esa exploración tiende a cero, permitiéndole concentrar casi toda la
campaña (98%) en C. La causa estructural de la diferencia es: el terco mantiene su tasa
de exploración constante para siempre, mientras que el apostador la deja decaer una
vez que el aprendizaje ya se consolidó. Uno sigue pagando el costo de explorar
indefinidamente, el otro solo mientras hace falta.


* **(c) (6 pts)** Tarea de ejecución: cambien VISITANTES = 5000 por 50000 y corran. Reporten
las compras de cada política y la diferencia entre ambas. ¿La desventaja del terco se cerró
o creció al darle diez veces más visitantes? Conecten el resultado con la lección de la clase:
“el terco nunca deja de perder”.
    -   R/            
        |   | Visitantes = 5000 | Visitantes = 50000 |
        | :--- | :--- | :--- |
        |Terco | 1 939 compras, 91.9% vio C | 19 374 compras, 93.1% vio C |
        | Apostador | 1 978 compras, 97.6% vio C | 19 948 compras, 99.7% vio C |
        | Diferencia (apostador - terco) | 39 compras | 574 compras|

    La desventaja del terco no se cerró, sino que creció. La diferencia en compras pasó de 39 (con
5 000 visitantes) a 574 (con 50 000 visitantes), casi 15 veces más, aunque los visitantes solo se
multiplicaron por 10. El terco se estancó en 93% de tráfico a C, mientras el apostador siguió
subiendo a 99.7%. Esto confirma "el terco nunca deja de perder": su regla fija no se corrige con
más datos, así que cuanto más dura la campaña, más compras absolutas pierde por seguir
explorando a ciegas. Su falla no es de datos ni de tiempo, es de diseño, una regla de
exploración fija no se corrige con más muestras, solo hace que el costo acumulado de esa
regla sea cada vez mayor.

## 2 Parte B: análisis conceptual y de código (40 puntos)

### B1. El modelo de tiempos de vuelo (14 pts)

Un analista ajustó el modelo t = θ · d (tiempo de vuelo en minutos como proporción de la
distancia en km) sobre las 29 rutas de rutas.csv, minimizando el error cuadrático medio con
descenso de gradiente. Probó dos tasas de aprendizaje. Este es su código y su salida real:

In [2]:
import numpy as np

d = np.array([234, 463, 236, 544, 1358, 470, 203, 296, 855, 361, 696, 496, 161, 816, 345, 321, 1161, 1168, 539, 1018, 420, 178, 262, 255, 652, 210, 728, 172, 558], dtype=float)
t = np.round(d / 800 * 60)  # minutos de vuelo observados

def gradiente(theta):
    return -2 * np.mean(d * (t - theta * d))  # derivada del error cuadrático medio

for lr, nombre in [(3e-7, "tasa A = 0.0000003"), (4e-6, "tasa B = 0.000004")]:
    theta = 0.0
    print(f"--- {nombre} ---")
    print(f"{'iteración':>10}{'theta':>12}{'error medio':>15}")
    for k in range(1, 41):
        theta = theta - lr * gradiente(theta)
        if k in (1, 2, 3, 5, 10, 20, 40):
            err = np.mean((t - theta * d) ** 2)
            print(f"{k:>10}{theta:>14.5g}{err:>15.4g}")

--- tasa A = 0.0000003 ---
 iteración       theta    error medio
         1      0.017141           1276
         2      0.030366          759.7
         3       0.04057          452.3
         5      0.054517          160.3
        10      0.069424          12.05
        20      0.074615         0.1354
        40      0.075032        0.06839
--- tasa B = 0.000004 ---
 iteración       theta    error medio
         1       0.22854           8972
         2      -0.23902      3.755e+04
         3       0.71753      1.572e+05
         5        2.7642      2.753e+06
        10       -96.299      3.536e+09
        20   -1.2378e+05      5.834e+15
        40   -2.0421e+11      1.588e+28


### Se pide

*   **(a) (4 pts)** El valor final con la tasa A es θ ≈ 0.0750. Interpreten físicamente ese número:
¿qué velocidad de crucero implica? Verifiquen con una cuenta.
    -   R/ El valor de θ ≈ 0.0750 representa el tiempo, en minutos, que le toma a la aeronave recorrer exactamente un kilómetro. Actúa como la pendiente de un modelo de regresión lineal simple donde el tiempo de vuelo es función de la distancia (Tiempo = θ × Distancia).

    -   Velocidad de crucero implicada: Implica una velocidad de crucero de 800 km/h.
    -   Verificación: Si la aeronave tarda 0.075 minutos por cada kilómetro recorrido, calculamos su velocidad invirtiendo el valor para obtener kilómetros por minuto, y luego multiplicamos por 60 para llevarlo a horas:
        -   Velocidad = 1 / 0.075 = 13.333 km/min
        -   13.333 km/min × 60 min/hora = 800 km/h
    
    Podemos comprobar esto usando una ruta de la tabla original, por ejemplo, de Guatemala a Panamá (GUA-PTY):

    -   Distancia: 1,358 km.
    - Tiempo estimado = 1358 km × 0.075 min/km = 101.85 minutos.
    Esto coincide de manera casi exacta con los 102 minutos de tiempo de vuelo estipulados en rutas.csv para ese tramo.

*   **(b) (5 pts)** Diagnostiquen qué pasó con cada tasa usando evidencia específica de la salida
(señalen números concretos). Presten atención al signo de θ en las primeras iteraciones de
la tasa B.
    -   R/ Tasa A (Convergencia exitosa): Con esta tasa (0.0000003), el algoritmo logró descender correctamente hacia el mínimo de la función de costo.

    -   Evidencia: El error medio se reduce de manera constante y monótona, cayendo desde 1276 en la iteración 1 hasta casi desaparecer en 0.06839 para la iteración 40, simultáneamente, el valor de θ se ajusta suavemente sin saltos bruscos, subiendo desde 0.017141 y estabilizándose en 0.075032.

    -   Tasa B (Divergencia por exceso de tamaño de paso): La tasa B (0.000004) resultó ser demasiado grande, en lugar de descender hacia el mínimo, los pasos tan amplios causaron que el algoritmo sobrepasara el fondo del "valle" del error una y otra vez, rebotando cada vez más alto y alejándose por completo de la solución.

    -   Evidencia del error: El error explotó de forma exponencial, comenzando alto (8972 en la iteración 1) y escalando a niveles absurdos de $1.588 \times 10^{28}$ en la iteración 40.

    -   Evidencia del signo de θ: En las primeras iteraciones, θ oscila violentamente de positivo a negativo (Iteración 1: 0.22854 → Iteración 2: -0.23902 → Iteración 3: 0.71753), ese cambio de signo refleja cómo el algoritmo rebota de un extremo al otro de la curva de error. Físicamente, el valor negativo de la iteración 2 es un absurdo que confirma el colapso del modelo, ya que indicaría que al recorrer una distancia positiva, el avión consumiría tiempo negativo.

*   **(c) (5 pts)** Este problema tiene solución exacta de mínimos cuadrados en una línea de álgebra.
Den dos razones por las que en problemas reales de ciencia de datos se usa igual el
descenso de gradiente, y expliquen qué precaución de esta salida aplicaría en cualquier
red neuronal.
    -   R/ Aunque existe una solución matemática exacta basada en álgebra lineal para este problema, en la práctica profesional se prefiere utilizar el descenso de gradiente por dos razones fundamentales, la primera solución analítica exige calcular la inversa de la matriz de características, una operación con una complejidad computacional tan alta que saturaría la memoria y tomaría demasiado tiempo al trabajar con bases de datos masivas, el descenso de gradiente resuelve esto al procesar los datos en pequeños fragmentos, lo que lo hace altamente escalable y eficiente y la segunda solución de álgebra lineal sirve exclusivamente para regresiones lineales, mientras que el descenso de gradiente es un motor de optimización universal que ofrece la flexibilidad necesaria para entrenar modelos complejos y no lineales donde simplemente no existe una ecuación matemática cerrada.

        En cuanto a la precaución indispensable para cualquier red neuronal, el ejercicio anterior ilustra claramente el peligro de configurar una tasa de aprendizaje excesivamente alta, si no se controla este hiperparámetro de manera rigurosa, las actualizaciones en los pesos de las neuronas serán demasiado grandes, lo que provocaría un fenómeno de desbordamiento donde el algoritmo rebotará violentamente en lugar de descender hacia el mínimo de la función de pérdida, haciendo que el error explote y destruyendo por completo el proceso de entrenamiento del modelo.

### B2. La gira de inspección (14 pts)

Una avioneta de la aerolínea debe salir de SJO, inspeccionar los otros 5 aeropuertos de la subred
del ejercicio A1 exactamente una vez, y regresar a SJO. La avioneta vuela en línea recta entre
cualesquiera dos aeropuertos (distancia de gran círculo desde aeropuertos.csv). Se compararon
dos métodos; salida real:


```text
    FUERZA BRUTA: evaluo 120 giras posibles
        mejor gira: SJO - LIR - SAL - SAP - RTB - TGU - SJO
        largo total: 1909.2 km
    RECOCIDO SIMULADO: arranca con una gira al azar de 3032.6 km
        paso 1: gira actual 2714.4 km ( temperatura 490.0)
        paso 50: gira actual 2034.9 km ( temperatura 182.1)
        paso 100: gira actual 2018.5 km ( temperatura 66.3)
        paso 200: gira actual 1933.4 km ( temperatura 8.8)
        paso 400: gira actual 1909.2 km ( temperatura 0.2)
        mejor gira encontrada : SJO - TGU - RTB - SAP - SAL - LIR - SJO
        largo total: 1909.2 km , evaluando solo 400 giras

### Se pide
*   **(a) (5 pts)** Los dos métodos reportan 1909.2 km pero las giras impresas se ven distintas.
¿Encontraron soluciones distintas o la misma? Justifiquen.
    -   R/ Ambos métodos encontraron exactamente la misma solución matemática y física, solo que recorrida en sentidos opuestos, si observamos el orden de los aeropuertos, la ruta encontrada por el recocido simulado (SJO - TGU - RTB - SAP - SAL - LIR - SJO) es el reverso exacto de la ruta reportada por la fuerza bruta (SJO - LIR - SAL - SAP - RTB - TGU - SJO). Dado que el problema asume vuelos en línea recta (distancias de gran círculo), la distancia para volar de un punto a otro es simétrica; es decir, cuesta exactamente los mismos kilómetros ir de LIR a SAL que de SAL a LIR. Al tratarse de un circuito cerrado y simétrico, recorrer el polígono en dirección horaria o antihoraria representa la misma gira, lo que justifica por qué el largo total de 1909.2 km es idéntico en ambas salidas.

*   **(b) (5 pts)** En el recocido simulado, ¿qué papel juega la temperatura y por qué el método
acepta a veces una gira peor que la actual? ¿Qué pasaría si nunca lo hiciera?
    -   R/ En el recocido simulado, la "temperatura" actúa como un parámetro de control que regula la probabilidad de explorar opciones que a simple vista no parecen favorables, al inicio del proceso, cuando la temperatura es alta, el algoritmo es muy permisivo y acepta con facilidad realizar saltos hacia configuraciones de viaje que son peores, conforme el proceso avanza y el sistema se "enfría", la temperatura disminuye progresivamente, haciendo que el algoritmo se vuelva cada vez más estricto y solo acepte movimientos que mejoren la distancia.

        El método acepta ocasionalmente una gira peor que la actual con el propósito fundamental de escapar de los mínimos locales, en problemas matemáticos complejos, el espacio de soluciones está lleno de trampas: rutas que parecen ser las mejores de su vecindario, pero que no son la óptima global. Aceptar temporalmente una ruta más larga le permite al algoritmo dar un paso hacia atrás para salir de ese valle engañoso y poder explorar otras áreas del mapa donde podría esconderse la ruta verdaderamente más corta.

        Si el algoritmo nunca aceptara una solución peor, se comportaría como un simple método de búsqueda local (conocido como hill climbing o escalada de colinas), al actuar de esta manera, en el momento en que se tope con una ruta donde cualquier cambio inmediato resulte en una distancia mayor, el algoritmo pensará erróneamente que ya terminó su trabajo y se detendrá. En consecuencia, se quedaría atascado para siempre en una solución mediocre o subóptima, sin la capacidad de dar el salto necesario para descubrir la mejor ruta posible.

*   **(c) (4 pts)** Si la red creciera a 15 aeropuertos, la fuerza bruta tendría que evaluar 14! ≈
8.7 × 1010 giras. Con esa cifra y la evidencia de la salida, argumenten en tres líneas
cuándo se justifica una heurística y qué garantía se sacrifica a cambio.
    -   R/ Se justifica usar una heurística cuando el número de combinaciones explota a niveles inmanejables, como evaluar $8.7 \times 10^{10}$ giras, volviendo la fuerza bruta computacionalmente inviable, como demostró la salida anterior, la heurística ofrece una inmensa eficiencia al lograr encontrar la mejor ruta evaluando apenas 400 opciones en lugar de todas las posibles. A cambio de este ahorro masivo de tiempo y recursos, el método sacrifica la garantía matemática absoluta de encontrar siempre el óptimo global verdadero.
    

### B3. Preguntas conceptuales (12 pts, 4 pts cada una)

Respondan cada una en un párrafo breve

*   **(i)** En aeropuertos.csv, San Salvador (SAL) es el aeropuerto con más conexiones: 9. ¿Basta
ese dato para afirmar que es el nodo más importante de la red? Propongan otra noción
de importancia vista en el curso que podría dar un resultado distinto, y expliquen qué
mide.
    -   R/ No, tener 9 conexiones no es suficiente para afirmar que San Salvador es el nodo más importante de la red. El grado de un nodo es una medida de importancia local. Otra noción de importancia en el curso es la centralidad de intermediación. Esta medida cuantifica la frecuencia con la que un nodo actúa como “puente” en los caminos más cortos entre otros pares de nodos. Un aeropuerto como Panamá, que conecta el norte y el sur de Centroamérica, podría tener un grado menor pero una centralidad de intermediación mucho más alta, siendo crucial para la conectividad global de la red y, por lo tanto, más “importante” desde esa perspectiva.


*   **(ii)** La tabla de rutas de OpenFlights es una fotografía histórica: su fuente dejó de actualizarse
en 2014. Señalen una conclusión concreta de esta prueba que podría cambiar con
datos actuales y expliquen, en términos de modelado, por qué todo modelo hereda las
limitaciones de sus datos.
    -   R/ 2. Una conclusión concreta que podría cambiar es la ruta más corta entre Liberia y Roatán. Con datos de 2014, la ruta más corta pasa por San Salvador (LIR–SAL–RTB). Si hoy existiera una nueva ruta directa entre alguno de estos aeropuertos, el resultado del algoritmo de Dijkstra podría ser completamente distinto. Esto ilustra una limitación de los modelos, el modelo es un reflejo de los datos con los que fue construido. Cualquier modelo de red hereda las limitaciones de su fuente de datos; en este caso, omite toda la evolución de la red aérea posterior a 2014, por lo que sus predicciones y análisis de conectividad podrían estar desactualizados y no reflejar la realidad actual.

*   **(iii)** La ecuación de Bellman dice que el valor de una situación es lo que gano ahora más
el valor descontado de la situación en que quedo. Usen esa idea para explicar por qué
la decisión de mayor ganancia inmediata puede no ser la óptima, con un ejemplo de la
propia aerolínea (por ejemplo: diferir el mantenimiento de un avión, o saturar la ruta más
rentable).
    -   R/ 3. La ecuación de Bellman destaca que las decisiones óptimas consideran tanto la recompensa inmediata como el valor futuro del estado resultante, por ejemplo, la aerolínea podría decidir saturar la ruta más rentablee con toda su flota para maximizar la ganancia inmediata, sin embargo, esta decisión deja a la aerolínea en un estado futuro vulnerable, sin capacidad de aprovechar otras oportunidades ni de reaccionar ante una caída de demanda en esa ruta. La decisión óptima, según Bellman, sería asignar parte de la flota a esa ruta rentable y otra parte a rutas alternativas, sacrificando algo de ganancia inmediata, pero asegurando un valor futuro mayor, lo que maximiza el valor total a largo plazo.

